# Ricerca di Short Tandem Repeats

Sviluppare un notebook per migliorare la qualità e individuare regioni a bassa complessità di
un dataset FASTQ.
Precisamente, si richiede di applicare in sequenza:
1. Un filtro “Sliding Window Quality” per effettuare il trimming del read al 3’, che consiste
nell’usare una finestra scorrevole lunga qualche base (parametro in input) e troncare
opportunamente il read al 3’ non appena la qualità media all’interno della finestra scende al
di sotto di una certa soglia (parametro in input)
2. Un filtro sulla lunghezza, per rimuovere i reads che sono troppo corti

Si richiede inoltre di riconoscere, all’interno dei reads, rimasti alla fine del precedente filtraggio, gli
**Short Tandem Repeats** (STRs) di lunghezza 1, 2 e 3. Uno Short Tandem Repeat è una sottostringa massimale in cui un certo motivo (lungo qualche base)
si ripete un certo numero di volte.

Esempio 1: GGGGGGGGGGGGGGGGGG (il motivo G di una base è ripetuto 18 volte)

Esempio 2: TATATATATATATATA (il motivo TA di due basi è ripetuto 8 volte)

Esempio 3: CAGCAGCAGCAGCAGCAGCAG (il motivo CAG di tre basi è ripetuto 7 volte)

Per ogni read (rimasto dopo il filtraggio), produrre in standard output gli eventuali STR, specificando
per ognuno la posizione di inizio nel read, il motivo e quante volte si ripete.

## 0) Importazione dei moduli necessari

In [1]:
from Bio import SeqIO, Seq

In [2]:
import numpy as np

In [3]:
import re

## 1) Parametri di input

E' necessario definire:
- La dimensione della sliding window espressa in basi
- La soglia di qualità di ogni sottostringa del read analizzata dalla sliding window
- La dimensione minima dei read dopo che sono stati filtrati dalla sliding window
- La lunghezza degli *Short Tandem Repeats* che verranno riconosciuti

In [4]:
sliding_window_dim = 5

In [5]:
quality_threshold = 10

In [6]:
read_min_dim = 5

In [7]:
fastq_file_path = "./"

In [8]:
str_lengths = [1, 2, 3]

## 2) Lettura del dataset

Il primo passaggio da svogere è caricare i read contenuti nel file FASTQ. 

In [9]:
records_generator = SeqIO.parse("./dataset-quality.fastq", "fastq")

## 3) Definizione del filtro "Sliding Window"

Il filtro *sliding window* viene implementato da una funzione che, dati in input:
- La sequenza del read da analizzare
- La dimensione della finestra scorrevole
- La soglia di qualità minima di ogni finestra

Restituisce il read troncato all'utima finestra con qualità maggiore alla soglia. 

In [10]:
def filter_by_quality(sequence, sliding_window_dim, quality_threshold):
    read_quality = sequence.letter_annotations["phred_quality"]
    for i in range(len(sequence) - sliding_window_dim):
        j = i + sliding_window_dim

        sequence_window = sequence.seq[i:j]
        qualities = read_quality[i:j]
        mean_quality = float(np.mean(qualities))

        if mean_quality < quality_threshold:
            return sequence[:i]

    return sequence

## 4) Definizione del filtro sulla lunghezza del read

Questo filtro viene implementato da una funzione che verifica che la lunghezza del read sia maggiore della soglia fornita.


In [11]:
def filter_by_length(sequence, read_min_dim):
    return len(sequence.seq) >= read_min_dim

## 5) Riconoscimento degli Short Tandem Repeats

Il read viene analizzato con una finestra scorrevole della dimensione del motivo da analizzare. Fino a quando il motivo viene ripetuto, si tratta di un short tandem repeat. 

In [19]:
def get_short_tandem_repeats(sequence: str):
    strs = []
    start_position = 0
    while start_position < len(sequence):
        position_strs = []
        for window_dim in [1, 2, 3]:
            end_position = start_position + window_dim - 1
            
            pattern = sequence[start_position:end_position+1]

            j = end_position + 1
            while sequence[j:j + window_dim] == pattern:
                j += window_dim

            if j != end_position + 1:
                str_pattern = sequence[start_position:end_position+1]
                str_start_position = start_position
                str_end_position = j - 1
                str_dim = str_end_position - str_start_position

                str_dict = {
                    "pattern": str_pattern,
                    "start": str_start_position,
                    "end": str_end_position
                }
                
                str_already_exists = False
                position_strs.append(str_dict)

        if position_strs:
            max_str = max(position_strs, key=lambda x: x["end"]-x["start"])
            strs.append(max_str)
    
            start_position += max_str["end"]-max_str["start"] + 1
        else:
            start_position += 1
    return strs

get_short_tandem_repeats("AAAAATAAAAA")

[{'pattern': 'A', 'start': 0, 'end': 4},
 {'pattern': 'A', 'start': 6, 'end': 10}]

## 6) Analisi dei reads contenuti nel file

Per ogni read contenuto nel file in input, applicare i filtri sulla qualità e sulla lunghezza. Infine, vengono trovati gli *short tandem repeats* sui read rimasti. 

In [ ]:
# FIXME rimuovi ricalcolo dei records
records_generator = SeqIO.parse("./dataset-quality.fastq", "fastq")
for record in records_generator:
    filtered_sequence = filter_by_quality(record, sliding_window_dim, quality_threshold)
    if filter_by_length(filtered_sequence, read_min_dim):
        records_strs = get_short_tandem_repeats(filtered_sequence.seq)
        print("Sequenza:", filtered_sequence.seq)
        for str_dict in records_strs:
            repeats_number = (str_dict["end"] + 1 - str_dict["start"]) / len(str_dict["pattern"])
            print(f"- {str_dict["pattern"]} ({str_dict["start"]}:{str_dict["end"]}) si ripete per {repeats_number} volte")